<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 1 (Lunes) — Predicción — Regresión lineal**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 🛒 Regresión lineal: volumen de ventas de productos (Whitewell)

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Predecir el volumen de ventas de productos nuevos a partir de las reseñas de clientes. |
| 🔍 Comprensión de los datos | Cargamos `productos_existentes.csv`, buscamos nulos y estudiamos correlaciones (multicolinealidad). |
| 🧹 Preparación de los datos | Dividimos en entrenamiento y prueba y validamos los rangos de cada conjunto. |
| 🤖 Modelado | Entrenamos una regresión lineal con `scikit-learn` y revisamos sus coeficientes. |
| 📊 Evaluación | Medimos el error cuadrático medio y el R² sobre el conjunto de prueba. |
| 🚀 Despliegue y conclusiones | Aplicamos el modelo a `nuevos_productos.csv` para estimar su volumen. |

> 🎯 **Analogía:** un gerente de tienda quiere saber cuánto venderá un producto nuevo mirando cómo se comportaron productos parecidos.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 👉 **1** | Lunes | **Predicción: regresión lineal y series de tiempo** |
| 2 | Martes | Clasificación 1: árboles de decisión, bosques aleatorios y KNN |
| 3 | Miércoles | Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales |
| 4 | Jueves | No supervisado: agrupación y asociación |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

### 📥 Datos del curso
Los archivos se descargan automáticamente (una sola vez) desde el repositorio del curso. Si ya los subió a `sample_data/`, no se vuelven a descargar.

In [ ]:
import os, urllib.request
REPO = 'https://gitlab.com/ejimenez/fiec_intro_machine_learning/-/raw/main'
os.makedirs('sample_data', exist_ok=True)
for nombre, dia in [('productos_existentes.csv', 'dia_1'), ('nuevos_productos.csv', 'dia_1')]:
    ruta = f'sample_data/{nombre}'
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(f'{REPO}/{dia}/{nombre}', ruta)
        print('Descargado:', ruta)
    else:
        print('Ya existe:', ruta)

---
## 🎯 Fase 1 · Comprensión del negocio

Predecir el volumen de ventas de productos nuevos a partir de las reseñas de clientes.

📚 Lectura recomendada: [3]

## Análisis del Problema

Whitewell ha sido un exitoso minorista de productos electrónicos durante más de 40 años, con más de 30 tiendas en el sudeste. Hace poco más de un año lanzo un nuestro sitio web de comercio electrónico. Estamos comenzando a acumular datos de transacciones de clientes del sitio y queremos aprovechar estos datos para informar nuestras decisiones sobre actividades relacionadas con el sitio, como marketing en línea, mejoras en el sitio, etc., para continuar maximizando la cantidad de ingresos que generamos de las ventas de comercio electrónico.
El equipo de ventas está considerando agregar algunos productos nuevos a la mezcla de productos de Whitewell. Han seleccionado 17 que se ajustan a la estrategia comercial de Blackwell, pero ahora necesitan ayuda para reducir la lista a cinco. Me gustaría ayudar al equipo de ventas prediciendo la rentabilidad de cada uno de los nuevos productos potenciales

 # Investigar  los patrones de la rentabilidad de los productos de Whitewell

Los modelos lineales son una de las herramientas más importantes del análisis cuantitativo. Los utilizamos cuando queremos predecir –o explicar– una variable dependiente a partir de una variable independiente. El uso mismo de la terminología de variables dependientes e independientes nos da una pista del abordaje: el ajuste de modelos lineales busca encontrar una función –en el sentido matemático – que nos permite expresar a una variable a partir de otras.

- Regresión Lineal (LM)

### Autores
   * Efrén Jiménez

### Librerías

In [ ]:
#Manejo de datasets
import pandas as pd
#Manejo de datasets
import numpy as np
#Gráficos
import matplotlib.pyplot as plt
#Gráficos
%matplotlib inline
#Gráficos basados en matplotlib
import seaborn as sns
from sklearn import linear_model
from sklearn.metrics import mean_squared_error, r2_score
#Ignorar Warnings
import warnings
warnings.filterwarnings('ignore')

import plotly.express as px
import plotly.graph_objects as go

---
## 🔍 Fase 2 · Comprensión de los datos

Cargamos `productos_existentes.csv`, buscamos nulos y estudiamos correlaciones (multicolinealidad).

📚 Lectura recomendada: [5] [6]

## Entendimiento de los Datos

El total es de 64 registros con 18 columnas dentro del set de datos de Whitewell

- ProductType   : variable de tipo cualitativa nominal
- Product       : variable de tipo cualitativa nominal
- Price         : variable de tipo cuantitativa continua
- X5StarReviews : variable de tipo cuantitativa
- X4StarReviews : variable de tipo cuantitativa
- X3StarReviews : variable de tipo cuantitativa
- X2StarReviews : variable de tipo cuantitativa
- X1StarReviews : variable de tipo cuantitativa
- PositiveServiceReview : variable de tipo cuantitativa
- NegativeServiceReview : variable de tipo cuantitativa
- WouldConsumerRecommendProduct : variable de tipo cuantitativa
- BestSellersRank : variable de tipo cuantitativa
- ShippingWeightlbs : variable de tipo cuantitativa
- ProductDepth : variable de tipo cuantitativa
- ProductWidth : variable de tipo cuantitativa
- ProductHeight: variable de tipo cuantitativa
- ProfitMargin : variable de tipo cuantitativa
- Volume : variable de tipo cuantitativa

In [ ]:
#Cargar los datos
productos = pd.read_csv(r"sample_data/productos_existentes.csv")
#Cargar los datos nuevos
productos_nuevos = pd.read_csv(r"sample_data/nuevos_productos.csv")

In [ ]:
#Primeros registros
productos.head()

In [ ]:
#Describir dataset
productos.describe()

In [ ]:
#Información del dataset
productos.info()

In [ ]:
# Numero de registros del dataset
print("Cantidad de registros:",len(productos))
# Cantidad de columnas del dataset
print("Cantidad de columnas:",len(productos.columns))
# Tipos de datos del dataset
print("Tipo de datos:",productos.dtypes.unique())
# Tamaño del daset
print("Filas y columnas:",productos.shape)


In [ ]:
# Chekear valores nulos
print(productos.isnull().any().sum(), ' / ', len(productos.columns))

In [ ]:
#Cuales columnnas estas con valores nulos
productos.isnull().sum()[productos.isnull().sum() > 0]

In [ ]:
# N/A= Valores vacios
#Usar solo si se quiere eliminar los registros que contienen columnas vacías
productos=productos.dropna()

**Multicolinealidad**

https://economipedia.com/definiciones/multicolinealidad.html#:~:text=La%20multicolinealidad%20es%20la%20relaci%C3%B3n,m%C3%A1s%20de%20dos%20variables%20explicativas.


![Mapa de calor de correlaciones](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch10/images/10_04.png)
*Figura: así se lee un mapa de calor de correlaciones (valores cercanos a ±1 = relación fuerte). [8]*

In [ ]:
productos.corr(numeric_only=True)

In [ ]:
corr = productos.corr(numeric_only=True)
sns.heatmap(corr)

### Tipos de relación
- -1,00 = Correlación negativa perfecta
- -0,90 = Correlación negativa muy fuerte
- -0,75 = Correlación negativa considerable
- -0,50 = Correlación negativa media
- -0,10 = Correlación negativa débil
- 0,00  = No existe correlación lineal alguna entre las variables
- 0,10  = Correlación positiva débil
- 0,50  = Correlación positiva media
- 0,75 = Correlación positiva considerable
- 0,90  = Correlación positiva muy fuerte
- 1,00  = Correlación positiva perfecta

---
## 🧹 Fase 3 · Preparación de los datos

Dividimos en entrenamiento y prueba y validamos los rangos de cada conjunto.

📚 Lectura recomendada: [4]

In [ ]:
#Dividir el conjunto de datos en uno de entrenamiento y otro de pruebas:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(productos[["PositiveServiceReview","X4StarReviews"]],productos[["Volume"]], test_size=0.3, random_state=1234)

>Es importante siempre validar los rangos de los conjuntos de datos creados, para evitar caer en extrapolación:

In [ ]:
print("Entrenamiento dimensiones",X_train.shape)
print("Descripción")
X_train.describe

In [ ]:
print("Pruebas dimensiones",X_train.shape)
print("Descripción")
X_train.describe

---
## 🔍 Fase 2 · Comprensión de los datos — relación entre variables seleccionadas

Cargamos `productos_existentes.csv`, buscamos nulos y estudiamos correlaciones (multicolinealidad).

In [ ]:
sns.pairplot(productos[["PositiveServiceReview","Volume","X4StarReviews"]], diag_kind="Volume", markers="+",
                  plot_kws=dict(s=50, edgecolor="b", linewidth=2),height=4, aspect=2,
                  diag_kws=dict(shade=True))
#sns.pairplot(productos, diag_kind="Volume", markers="+",
#                  plot_kws=dict(s=50, edgecolor="b", linewidth=1),
#                  diag_kws=dict(shade=True))
plt.show()

In [ ]:
# Crear el gráfico de correlación
sns.scatterplot(x="PositiveServiceReview", y="X4StarReviews", size="Volume",
                data=(productos[["PositiveServiceReview","Volume","X4StarReviews"]]))

In [ ]:
# Crear el gráfico de correlación
fig = px.scatter(productos[["PositiveServiceReview","Volume","X4StarReviews"]], x="PositiveServiceReview", y="X4StarReviews", size="Volume",
                 title="Gráfico de Correlación")

# Mostrar el gráfico
fig.show()

In [ ]:
# Crear el gráfico de correlación en 3D
fig = go.Figure(data=[go.Scatter3d(
    x=productos['PositiveServiceReview'],
    y=productos['X4StarReviews'],
    z=productos['Volume'],
    mode='markers',
    marker=dict(
        size=12,
        color=productos['Volume'],
        colorscale='Viridis',
        opacity=0.8
    )
)])

# Añadir etiquetas a los ejes y título
fig.update_layout(scene=dict(
    xaxis_title='PositiveServiceReview',
    yaxis_title='X4StarReviews',
    zaxis_title='Volume'),
    title='Gráfico de Correlación en 3D')

# Mostrar el gráfico
fig.show()

---
## 🤖 Fase 4 · Modelado

Entrenamos una regresión lineal con `scikit-learn` y revisamos sus coeficientes.

📚 Lectura recomendada: [4]

Una vez seleccionadas las variables para incluir en el modelo de regresión, se procede a crearlo:

##### Regresion Lineal

In [ ]:
#Generar el modelo de regresión lineal
regr = linear_model.LinearRegression()

In [ ]:
# Entrenamos nuestro modelo
regr.fit(X_train, y_train)

In [ ]:
# Hacemos las predicciones que en definitiva una línea (en este caso, al ser 2D)
y_pred = regr.predict(X_test)
y_pred

In [ ]:
print("Mean squared error: %.2f" % mean_squared_error(y_test, y_pred))

In [ ]:
# Veamos los coeficienetes obtenidos, En nuestro caso, serán la Tangente
print('Coefficients: \n', regr.coef_)
# Este es el valor donde corta el eje Y (en X=0)
print('Independent term: \n', regr.intercept_)
# Error Cuadrado Medio
print("Mean squared error: %.2f" % mean_squared_error(y_test, y_pred))
# Puntaje de Varianza. El mejor puntaje es un 1.0
print('Variance score: %.2f' % r2_score(y_test, y_pred))

---
## 📊 Fase 5 · Evaluación

Medimos el error cuadrático medio y el R² sobre el conjunto de prueba.

📚 Lectura recomendada: [3]

Calidad del modelo de machine learning

In [ ]:
regr.score(X_test, y_test)

---
## 🚀 Fase 6 · Despliegue y conclusiones

Aplicamos el modelo a `nuevos_productos.csv` para estimar su volumen.

📚 Lectura recomendada: [4]

In [ ]:
#Predicciones
prediccion=regr.predict(productos_nuevos[["PositiveServiceReview","X4StarReviews"]])

validate=np.hstack((productos_nuevos[["Product"]],np.around(prediccion,decimals=2)))
data=pd.DataFrame(validate, columns = ['Producto','valor_prediccion'])
data

## Resultados
En conclusión, los valores obtenidos por la modelación de regresión lineal simple en los datos que se estudiaron
pueden establecerse como buenos, debido a que el total de las observaciones que puede predecir correctamente es de alrededor
del 82% de la observaciones. Por lo que se recomienda utilizar el modelo generado para predecir el volumen de compras de la empresa

---
## 🧪 Refuerzo: regularización: Ridge y Lasso

Con predictoras correlacionadas, la regresión ordinaria puede sobreajustar. **Ridge** (penalización $\lambda\sum\beta_j^2$) encoge los coeficientes; **Lasso** (penalización $\lambda\sum|\beta_j|$) puede llevar algunos exactamente a 0, o sea, selecciona variables [3].

In [ ]:
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

for nombre, m in [('Lineal', LinearRegression()), ('Ridge', Ridge(alpha=1.0)), ('Lasso', Lasso(alpha=0.1, max_iter=10000))]:
    p = make_pipeline(StandardScaler(), m).fit(X_train, y_train)
    print(f'{nombre:7s} R² prueba = {p.score(X_test, y_test):.3f}')

### ✍️ Ejercicios
1. Varíe `alpha` en Ridge y Lasso (0.01, 1, 10): ¿cuándo empieza a empeorar el R²?
2. Con Lasso, imprima `p[-1].coef_`: ¿qué variables se eliminan?

---
## ✅ Conclusiones y siguientes pasos

Con regresión lineal predijimos una **variable numérica**. Enseguida la predicción depende del **tiempo** (series de tiempo) y el **Martes** pasamos a predecir **categorías** [3].

---
## 📚 Referencias

Fuentes adicionales para profundizar: [7].

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** James, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J. (2023). *An Introduction to Statistical Learning with Applications in Python*. Springer. https://doi.org/10.1007/978-3-031-38747-0

**[4]** Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html

**[5]** McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56–61. https://doi.org/10.25080/Majora-92bf1922-00a

**[6]** Waskom, M. L. (2021). seaborn: statistical data visualization. *Journal of Open Source Software, 6*(60), 3021. https://doi.org/10.21105/joss.03021

**[7]** Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering, 9*(3), 90–95. https://doi.org/10.1109/MCSE.2007.55

**[8]** Raschka, S., & Mirjalili, V. (2019). *Python Machine Learning* (3.ª ed.). Packt Publishing. (Figuras usadas desde https://github.com/rasbt/python-machine-learning-book-3rd-edition)